## Project: Manchu-English Translation, Attempt 4: LLM In-context Learning + Norman Dictionary

In [1]:
! pip install sacrebleu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 4.2 MB/s eta 0:00:00


In [4]:
import os
import json
import time
import re
import string
import pandas as pd
from typing import List

from sacrebleu.metrics import BLEU
from tqdm import tqdm

from google.colab import ai

In [5]:
SEED = 6600

DATA_PATH = "/content/drive/MyDrive/2025nn/parallel.csv"
DICT_PATH = "/content/drive/MyDrive/2025nn/norman_dict.csv"

SRC_COL = "manchu"
TGT_COL = "english"

OUT_DIR = "./runs/attempt4_incontext-manc"
os.makedirs(OUT_DIR, exist_ok=True)

OUT_CSV = os.path.join(OUT_DIR, "att4-manc-test_translations.csv")

MAX_DICT_ENTRIES = 8
MAX_EXAMPLES = None
SLEEP_SEC = 0.5

MODEL_NAME = "attempt4_incontext_dict"

# BLEU
bleu_sent = BLEU(effective_order=True)
bleu_corpus = BLEU()

#### BLEU is case-sensitive, normalize in the way did for parallel.csv

In [6]:
def normalize_text(s: str) -> str:
    if not isinstance(s, str):
        return ""
    s = s.lower()
    s = re.sub(rf"[{re.escape(string.punctuation)}]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


In [7]:
def load_splits():
    from sklearn.model_selection import train_test_split

    df = pd.read_csv(DATA_PATH)
    src_all = tuple(df[SRC_COL].astype(str))
    tgt_all = tuple(df[TGT_COL].astype(str))

    train_src, test_src, train_tgt, test_tgt = train_test_split(
        src_all, tgt_all, test_size=0.25, random_state=SEED
    )

    val_src, test_src, val_tgt, test_tgt = train_test_split(
        test_src, test_tgt, test_size=0.5, random_state=SEED
    )

    return test_src, test_tgt


def load_dictionary():
    df = pd.read_csv(DICT_PATH)
    df["item"] = df["item"].astype(str).str.upper()
    return dict(zip(df["item"], df["meaning"]))



### Build Prompt

references:

- *"Understanding In-Context Machine Translation for Low-Resource Languages: A Case Study on Manchu"* https://arxiv.org/abs/2502.11862


In [8]:
def retrieve_dict_entries(src_sentence: str, lexicon: dict, k=MAX_DICT_ENTRIES):
    words = [w.upper() for w in src_sentence.split()]
    hits = []
    for w in words:
        if w in lexicon:
            hits.append(f"{w}: {lexicon[w]}")
        if len(hits) >= k:
            break
    return hits

# adopt from the Pei et al.(2025) appendix
def build_prompt(src_sentence: str, dict_entries: List[str]) -> str:
    prompt = [
        "\nPlease help me translate the following input Manchu sentence into English",
        "For the translation task, you are given the word by word mapping",
        "from the source language words to the target language words",
        """
        Please try your best to translate, it’s okay if
        your translation is bad. Do not refuse to try it.
        I won’t blame you.
        """,
        # "Use the following dictionary entries to assist translation.\n.",
        "IMPORTANT:",
        "- Output the input sentence's English translation ONLY.",
        "",
        "Dictionary entries (Word or Phrase mapping to its English meaning):"
    ]

    if dict_entries:
        prompt.extend(dict_entries)
    else:
        prompt.append("(No relevant dictionary entries.)")

    prompt.extend([
        "",
        "### TRANSLATION ###",
        f"Input: {src_sentence}",
        "English:",
        "### END ###"
    ])

    return "\n".join(prompt)


## Call google.colab.ai for generation

In [9]:
def extract_translation(raw: str) -> str:
    """
    Extract translation between markers.
    Fallback: last non-empty line.
    """
    if "### TRANSLATION ###" in raw and "### END ###" in raw:
        chunk = raw.split("### TRANSLATION ###")[-1]
        chunk = chunk.split("### END ###")[0]
        return chunk.replace("English:", "").strip()

    lines = [l.strip() for l in raw.splitlines() if l.strip()]
    return lines[-1] if lines else ""


def translate_in_context(src_sentence: str, lexicon: dict) -> str:
    prompt = build_prompt(src_sentence, retrieve_dict_entries(src_sentence, lexicon))
    print(prompt)
    response_raw = ""
    print('-' * 20)
    print(f'\nSource: {src_sentence}')

    # call google.colab.ai
    try:
        response_raw = ai.generate_text(prompt)
        print(response_raw)
    except TypeError as e:
        print(f"Warning: '{src_sentence}' failed with error: {e}")

    except Exception as e:
        print(f"Warning: failed at '{src_sentence}': {e}")

    # extract translation
    return extract_translation(response_raw)


def sentence_bleu(hyp: str, ref: str) -> float:
    if hyp.strip() == "":
        return 0.0
    return bleu_sent.sentence_score(hyp, [ref]).score


### Main

In [10]:
def main():
    test_src, test_tgt = load_splits()
    lexicon = load_dictionary()

    rows = []
    raw_log_path = os.path.join(OUT_DIR, "raw_generations.jsonl")

    with open(raw_log_path, "w", encoding="utf-8") as logf:
        for i, (src, ref) in enumerate(tqdm(zip(test_src, test_tgt), total=len(test_src))):
            if MAX_EXAMPLES and i >= MAX_EXAMPLES:
                break

            gen_raw = translate_in_context(src, lexicon)

            # normalization (for BLEU only)
            # ref_norm = normalize_text(ref)
            gen_norm = normalize_text(gen_raw)

            sbleu = sentence_bleu(gen_norm, ref)

            rows.append({
                "id": i,
                "manchu": src,
                "reference": ref,
                "generated": gen_raw,
                # "reference_norm": ref_norm,
                "generated_norm": gen_norm,
                "sentence_bleu": sbleu,
                "model": MODEL_NAME,
            })

            logf.write(json.dumps({
                "id": i,
                "src": src,
                "ref": ref,
                "gen": gen_raw,
                "gen_norm": gen_norm,
                "sentence_bleu": sbleu,
            }, ensure_ascii=False) + "\n")

            time.sleep(SLEEP_SEC)

    df_out = pd.DataFrame(rows)
    df_out.to_csv(OUT_CSV, index=False, encoding="utf-8")

    # corpus bleu on testset for comparison
    corpus_bleu_score = bleu_corpus.corpus_score(
        df_out["generated_norm"].tolist(),
        [df_out["reference"].tolist()]
    )

    print("\n" + "=" * 70)
    print("ATTEMPT 4 IN-CONTEXT MT (DICTIONARY)")
    print("=" * 70)
    print("Test corpus BLEU:")
    print(corpus_bleu_score)
    print("\nSaved CSV here:", OUT_CSV)
    print(df_out.head())


if __name__ == "__main__":
    main()

  0%|          | 0/38 [00:00<?, ?it/s]


Please help me translate the following input Manchu sentence into English
For the translation task, you are given the word by word mapping
from the source language words to the target language words

        Please try your best to translate, it’s okay if
        your translation is bad. Do not refuse to try it.
        I won’t blame you.
        
IMPORTANT:
- Output the input sentence's English translation ONLY.

Dictionary entries (Word or Phrase mapping to its English meaning):
MANGGAI: merely, simply
AMBA: big, great, vast, important
TACIN: learning, science, skill; religion; custom, habit
I: he, she; the genitive particle; an interjection used to get the attention of subordinates
BITHE: book; letter
DABALA: (sentence particle) only, merely; (postposition) besides

### TRANSLATION ###
Input: manggai amba tacin i bithe dabala
English:
### END ###
--------------------

Source: manggai amba tacin i bithe dabala
Merely a book of great learning.


  3%|▎         | 1/38 [00:09<05:41,  9.22s/it]


Please help me translate the following input Manchu sentence into English
For the translation task, you are given the word by word mapping
from the source language words to the target language words

        Please try your best to translate, it’s okay if
        your translation is bad. Do not refuse to try it.
        I won’t blame you.
        
IMPORTANT:
- Output the input sentence's English translation ONLY.

Dictionary entries (Word or Phrase mapping to its English meaning):
BI: I, me; there is, there are, has, have
UMAI: (not) at all, totally, entirely

### TRANSLATION ###
Input: bi umai seme gisurehekv
English:
### END ###
--------------------

Source: bi umai seme gisurehekv
I did not speak at all.


  5%|▌         | 2/38 [00:14<03:58,  6.64s/it]

...


Please help me translate the following input Manchu sentence into English
For the translation task, you are given the word by word mapping
from the source language words to the target language words

        Please try your best to translate, it’s okay if
        your translation is bad. Do not refuse to try it.
        I won’t blame you.
        
IMPORTANT:
- Output the input sentence's English translation ONLY.

Dictionary entries (Word or Phrase mapping to its English meaning):
ERE: this
INU: also, too; even (adverb); so, yes; correct
OMO: lake, pond
DE: the dative-locative particle
EMU: one
ADALI: like, same
DABKVRI: double, having layers, storied (building)
NIO: an interrogative sentence particle; an emphatic particle

### TRANSLATION ###
Input: ere inu musei booi omo de bihe ilhai emu adali dabkvri jeringge nio
English:
### END ###
--------------------

Source: ere inu musei booi omo de bihe ilhai emu adali dabkvri jeringge nio
Is this also like one magnificent storied building 

 97%|█████████▋| 37/38 [07:29<00:15, 15.77s/it]


Please help me translate the following input Manchu sentence into English
For the translation task, you are given the word by word mapping
from the source language words to the target language words

        Please try your best to translate, it’s okay if
        your translation is bad. Do not refuse to try it.
        I won’t blame you.
        
IMPORTANT:
- Output the input sentence's English translation ONLY.

Dictionary entries (Word or Phrase mapping to its English meaning):
SI: you (singular); space (in writing); cf. siden; a file of five men; obstruction, blocking
MIMBE: accusative of hi: me
UME: verb used for negating imperatives (stands before the imperfect participle)

### TRANSLATION ###
Input: si mimbe ume holtoro
English:
### END ###
--------------------

Source: si mimbe ume holtoro
Do not deceive me.


100%|██████████| 38/38 [07:33<00:00, 11.93s/it]


ATTEMPT 4 IN-CONTEXT MT (DICTIONARY)
Test corpus BLEU:
BLEU = 5.31 31.6/9.6/2.8/1.1 (BP = 0.949 ratio = 0.950 hyp_len = 288 ref_len = 303)

Saved CSV here: ./runs/attempt4_incontext-manc/att4-manc-test_translations.csv
   id                             manchu  \
0   0  manggai amba tacin i bithe dabala   
1   1            bi umai seme gisurehekv   
2   2       doron be waliyabuci ja baita   
3   3  xi halai nun simbe aliyame tehebi   
4   4        jiduji loo tai tai fulu kai   

                                           reference  \
0                           it is the great learning   
1                          i was not saying anything   
2  the loss of a seal is an ordinary occurrence a...   
3                     cousin shih is waiting for you   
4                            your venerable ladyship   

                                           generated  \
0                   Merely a book of great learning.   
1                            I did not speak at all.   
2  If the 